In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import joblib
import torch
import torch.nn as nn
import segmentation_models_pytorch as smp
import warnings
warnings.filterwarnings('ignore')

from src.data_utils import load_config, ensure_dir
from src.metrics import evaluate_all

plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

device = torch.device('cuda')
print('所有包导入成功')

所有包导入成功


In [2]:
cfg           = load_config('../configs/config.yaml')
processed_dir = Path('../data/processed')
model_dir     = Path('../models')
fig_dir       = ensure_dir('../results/figures')
report_dir    = ensure_dir('../results/reports')
LABEL         = '2024_04'

# 加载特征和标签
features = np.load(processed_dir / f'NSW_test_features_{LABEL}.npy')
labels   = np.load(processed_dir / f'NSW_test_label_ero_{LABEL.replace("_","")}.npy')

# 有效像素掩膜
valid_mask = np.isfinite(labels) & np.all(np.isfinite(features), axis=-1)
X = features[valid_mask]
y = labels[valid_mask]

# 采样测试集（固定随机种子保证可重复）
np.random.seed(42)
idx      = np.random.choice(len(X), size=200_000, replace=False)
X_test   = X[idx]
y_test   = y[idx]

print(f'特征形状  : {features.shape}')
print(f'标签形状  : {labels.shape}')
print(f'测试样本数: {len(X_test):,}')
print(f'y 范围    : {y_test.min():.4f} ~ {y_test.max():.4f} t/ha/月')

# 加载 Random Forest
rf = joblib.load(model_dir / 'rf_model_2024_04.joblib')
print('\nRandom Forest 加载成功')

# 加载 XGBoost
import xgboost as xgb
xgb_model = joblib.load(model_dir / 'xgb_model_2024_04.joblib')
print('XGBoost 加载成功')

# 加载 2D CNN
class CNN2D(nn.Module):
    def __init__(self, in_channels=7):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, 32, 3, padding=1), nn.ReLU(), nn.BatchNorm2d(32),
            nn.Conv2d(32, 64, 3, padding=1),          nn.ReLU(), nn.BatchNorm2d(64),
            nn.Conv2d(64, 128, 3, padding=1),         nn.ReLU(), nn.BatchNorm2d(128),
            nn.Conv2d(128, 1, 1)
        )
    def forward(self, x):
        return self.net(x).squeeze(1)

# 加载 U-Net
unet = smp.Unet(
    encoder_name='resnet34', encoder_weights=None,
    in_channels=7, classes=1, activation=None
).to(device)
unet.load_state_dict(torch.load(model_dir / 'unet_model_2024_04.pth'))
unet.eval()
print('U-Net 加载成功')

特征形状  : (16811, 18790, 7)
标签形状  : (16811, 18790)
测试样本数: 200,000
y 范围    : 0.0505 ~ 579.1266 t/ha/月

Random Forest 加载成功
XGBoost 加载成功
U-Net 加载成功
